# Damped Harmonic Oscillator — Physics-Informed Neural Network

This notebook trains a PINN to solve the 1D damped harmonic oscillator and
demonstrates the **gradient-flow inspection** tooling that is the centrepiece of
this repo.

Governing ODE:  $m\,x'' + c\,x' + k\,x = 0$,  with $x(0)=x_0$, $x'(0)=v_0$.

There is no dataset to download — the "data" is a cloud of collocation points
sampled from the time domain, and the physics is enforced through the loss.
Everything below runs on a laptop CPU in well under a minute.

**To plug in your own problem:** subclass `BaseDataset` and `BaseModel`, edit
`configs/default.yaml`, and re-run. The training loop, MLflow logging,
checkpointing, and gradient visualization are unchanged.

In [ ]:
%load_ext autoreload
%autoreload 2

from src.config import Config
from src.dataset import OscillatorDataset, make_dataloaders
from src.model import MLP
from src.trainer import Trainer
from visualizations import GradientTracker, plot_architecture

try:
    import mlflow
except Exception:
    mlflow = None

cfg = Config.from_yaml("../configs/default.yaml")
cfg.experiment.name

## 1. Build the dataset and model

In [ ]:
dataset = OscillatorDataset(cfg.physics)
train_loader, val_loader, test_loader = make_dataloaders(
    dataset, batch_size=256, val_split=0.15, test_split=0.15,
    num_workers=0, seed=cfg.experiment.seed,
)

model = MLP(cfg.model)
print(f"{model.num_parameters()} trainable parameters")

## 2. Architecture diagram

`plot_architecture` inspects the model at runtime and emits a Mermaid graph.
Paste the printed source into a Markdown cell fenced as ```mermaid``` to render
it, or view `runs/architecture.mmd` on GitHub.

In [ ]:
mermaid = plot_architecture(model, input_shape=(1, cfg.model.in_dim),
                            save_path="../runs/architecture.mmd")
print(mermaid)

## 3. Train

The `GradientTracker` records the per-layer gradient L2 norm at every step and
aggregates per epoch. The PINN loss combines a PDE-residual term and an
initial-condition term — watch how their imbalance shows up in the gradient
heatmap below.

In [ ]:
tracker = GradientTracker(model, metric=cfg.visualization.gradient_metric)

if mlflow is not None:
    mlflow.set_tracking_uri(cfg.mlflow.tracking_uri)
    mlflow.set_experiment(cfg.mlflow.experiment_name)
    run = mlflow.start_run(run_name=cfg.experiment.name)
    mlflow.log_params(cfg.to_flat_dict())

trainer = Trainer(model, dataset, cfg, tracker=tracker, mlflow_module=mlflow)
result = trainer.fit(train_loader, val_loader, test_loader, run_dir="../runs")
result

## 4. Gradient-flow inspection — the headline result

Three views of where and when learning happened across the network.

In [ ]:
tracker.plot_heatmap("../runs/gradient_heatmap.png");

In [ ]:
tracker.plot_curves("../runs/gradient_curves.png");

In [ ]:
tracker.plot_contributions("../runs/layer_contributions.png");

## 5. Did it learn the physics?

Compare the network prediction against the closed-form analytical solution.

In [ ]:
import numpy as np, torch
import matplotlib.pyplot as plt

model.eval()
t = np.linspace(0.0, cfg.physics.t_max, 500, dtype=np.float32)
with torch.no_grad():
    pred = model(torch.from_numpy(t).unsqueeze(1)).squeeze(1).numpy()
exact = dataset.analytical_solution(t)
mae = np.mean(np.abs(pred - exact))

plt.figure(figsize=(9,4))
plt.plot(t, exact, "k-", lw=2, label="analytical")
plt.plot(t, pred, "r--", lw=1.6, label="PINN")
plt.xlabel("t"); plt.ylabel("x(t)"); plt.legend()
plt.title(f"PINN vs analytical (MAE={mae:.3e})")
plt.grid(True, ls=":", alpha=0.4)
plt.savefig("../runs/solution_vs_analytical.png", dpi=130, bbox_inches="tight")
plt.show()
print("MAE:", mae)

In [ ]:
# Log artifacts and close the MLflow run
if mlflow is not None:
    for p in ["gradient_heatmap.png","gradient_curves.png","layer_contributions.png",
              "solution_vs_analytical.png","architecture.mmd"]:
        mlflow.log_artifact(f"../runs/{p}")
    mlflow.end_run()
print("done")